# Анализ результатов экспериментов с Pruning

Простая загрузка моделей после pruning.

In [1]:
import os
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# Device selection
os.environ["CUDA_DEVICE_ORDER"] = 'PCI_BUS_ID'
os.environ["CUDA_VISIBLE_DEVICES"] = '2'  # Change GPU number here

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

Using device: cuda


In [2]:
def load_model(base_path, adapter_path=None):
    tokenizer = AutoTokenizer.from_pretrained(base_path)
    model = AutoModelForCausalLM.from_pretrained(
        base_path,
        torch_dtype=torch.float16,
        device_map='auto'
    )
    
    if adapter_path:
        model = PeftModel.from_pretrained(model, adapter_path)
    
    return model, tokenizer

In [3]:
# Пути к моделям
base_model_path = '../../src/checkpoints/llama3.1-8b'
window_adapter_path = '../../src/checkpoints/llama3.1-8b_p_window'
iterative_adapter_path = '../../src/checkpoints/llama3.1-8b_p_iter'

In [4]:
# Загрузка базовой модели
print('Загрузка базовой модели...')
base_model, tokenizer = load_model(base_model_path)
print(f'Базовая модель: {type(base_model).__name__}')

Загрузка базовой модели...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Базовая модель: LlamaForCausalLM


In [5]:
# Загрузка модели после window pruning
print('Загрузка модели после window pruning...')
window_model, _ = load_model(base_model_path, window_adapter_path)
print(f'Window pruned модель: {type(window_model).__name__}')

Загрузка модели после window pruning...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Window pruned модель: PeftModelForCausalLM


/home/ThunderstormXX/Ridiculous-LLM-Compression/.venv/lib/python3.10/site-packages/peft/peft_model.py:585: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.25.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.25.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.25.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.25.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.layers.25.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.layers.25.mlp.down_proj.lora_B.default.weight', 'base_model.model.model.layers.26.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.26.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.26.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.26.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.layers.26.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.layers.26.mlp.

In [6]:
window_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
              (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
              (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
              (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
            )
            (mlp): LlamaMLP(
              (gate_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=14336, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_feat

In [ ]:
# Загрузка модели после iterative pruning
print('Загрузка модели после iterative pruning...')
iterative_model, _ = load_model(base_model_path, iterative_adapter_path)
print(f'Iterative pruned модель: {type(iterative_model).__name__}')

In [ ]:
# Вывод структуры моделей
print('\n=== Структура базовой модели ===')
print(base_model)

print('\n=== Структура window pruned модели ===')
print(window_model)

print('\n=== Структура iterative pruned модели ===')
print(iterative_model)